# Syllabus-Grounded Contextual Biasing — full study on Kaggle

Kaggle suits this workload better than free Colab: **9-hour GPU sessions**, 30 GPU-hours
a week, and — the part that matters — `/kaggle/working` is saved as notebook output, so a
session ending does not destroy the decode cache.

## Settings to change before running (right-hand panel)

1. **Accelerator → GPU T4 x2** (or P100).
2. **Internet → On.** Needed for `pip`, `git clone` and the corpus download. Kaggle
   requires a phone-verified account to enable it.
3. **Persistence → Files only** (optional, helps within a session).

## How to survive session limits

Two habits, both cheap:

* **Run it headless with "Save Version → Save & Run All".** The notebook then executes on
  Kaggle's machines with the browser closed, up to the 9-hour GPU limit, and everything in
  `/kaggle/working` is committed as that version's output. This is the reliable path for the
  Tier-2 matrix.
* **Resume from the previous version's output.** After a run, use *File → Add Data →
  Notebook Output* (or the *Input* panel) to attach this notebook's own output to the next
  run. Cell 2 finds `checkpoints/asr_cache.tar.gz` under `/kaggle/input/` automatically and
  restores it, so completed decodes are never repeated.

Bulky, regenerable things (the tarball, the cut audio) go to `/kaggle/temp`, which is *not*
saved. The decode cache is tarred into `/kaggle/working/checkpoints/` after every stage, so
the output stays a handful of files rather than tens of thousands.

## Two rules that keep the results valid

1. **One results table, one platform.** GPU float16 and CPU int8 produce different
   hypotheses. `device` and `compute_type` are part of the cache key so they cannot mix
   inside a cache, but do not put a CPU row and a GPU row in the same table.
2. **The pilots belong to the platform that reports the numbers**, so cell 7 re-runs them
   here. They cost about fifteen minutes on a GPU.

In [ ]:
# 1. Environment check ---------------------------------------------------------
!nvidia-smi
import socket
try:
    socket.create_connection(('pypi.org', 443), timeout=6).close()
    print('\ninternet: ON')
except OSError:
    raise SystemExit('internet is OFF — enable it in Settings, or nothing below works')

In [ ]:
# 2. Layout, code, and restore from the last session ---------------------------
# /kaggle/working  is saved as notebook output  -> repo, runs/, report/, checkpoints/
# /kaggle/temp     is scratch, wiped on exit    -> tarball, cut audio, live cache
import os, glob, subprocess, pathlib, shutil

REPO = '/kaggle/working/FYP'
SCRATCH = '/kaggle/temp'
CKPT = pathlib.Path('/kaggle/working/checkpoints')
CKPT.mkdir(parents=True, exist_ok=True)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/meet244/FYP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
os.chdir(REPO)

# Keep audio and the live cache off the saved output.
for name in ('data', 'cache'):
    target = f'{SCRATCH}/{name}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(name):
        shutil.rmtree(name, ignore_errors=True)
        os.symlink(target, name)

os.environ['PYTHONPATH'] = 'src'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Restore the newest checkpoint from an attached previous output, if there is one.
cands = sorted(glob.glob('/kaggle/input/*/checkpoints/*.tar.gz')
               + glob.glob('/kaggle/input/*/*/checkpoints/*.tar.gz'))
if cands:
    for tgz in cands:
        print('restoring', tgz)
        subprocess.run(['tar', '-xzf', tgz], check=False)
    n = len(glob.glob('cache/asr/*/*/*.json'))
    print(f'{n} cached decodes restored — these will not be recomputed')
else:
    print('no previous checkpoint attached; starting from scratch')
print('cwd', os.getcwd())

In [ ]:
# 3. A checkpoint helper, used after every stage -------------------------------
# The decode cache is the expensive artefact. Tarring it into the saved output after
# each stage means a session that dies mid-matrix costs one condition, not the run.
def checkpoint(tag=''):
    import subprocess, glob
    subprocess.run(['tar', '-czf', str(CKPT / 'asr_cache.tar.gz'), 'cache/asr'],
                   check=False)
    subprocess.run(['tar', '-czf', str(CKPT / 'runs_report.tar.gz'),
                    'runs', 'report', 'configs', 'data/manifests'], check=False)
    n = len(glob.glob('cache/asr/*/*/*.json'))
    sz = sum(f.stat().st_size for f in CKPT.glob('*.tar.gz')) / 1e6
    print(f'[checkpoint {tag}] {n} decodes cached, {sz:.0f} MB in /kaggle/working')

def run(*args, tail=30):
    """Run a harness script, showing the last `tail` lines (progress bars stripped)."""
    import subprocess, sys
    p = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                       env={**os.environ, 'PYTHONPATH': 'src'})
    lines = [l for l in (p.stdout + p.stderr).replace('\r', '\n').splitlines()
             if 'utt/s' not in l and 'it/s]' not in l and l.strip()]
    print('\n'.join(lines[-tail:]))
    if p.returncode:
        raise SystemExit(f'{args[0]} failed with exit {p.returncode}')

print('helpers ready')

In [ ]:
# 4. Dependencies --------------------------------------------------------------
# requirements.txt pins CPU-oriented versions (numpy<2 etc.) that fight Kaggle's
# preinstalled torch, so install the minimum set and leave torch alone.
!pip install -q faster-whisper==1.1.1 "ctranslate2==4.5.0" jiwer==3.0.4 \
    rapidfuzz indic-transliteration sentence-transformers soundfile librosa \
    pyyaml matplotlib 2>&1 | tail -2
!python -c "import ctranslate2; print('ctranslate2', ctranslate2.__version__)"

In [ ]:
# 5. Make cuDNN visible, then prove a GPU decode works -------------------------
# CTranslate2 loads cuDNN from the NVIDIA pip wheels, not the image's default library
# path. Symptom when it cannot: 'Unable to load libcudnn_ops.so.9'.
import site, glob, os, subprocess, sys
libdirs = {p for sp in site.getsitepackages() + [site.getusersitepackages()]
           for p in glob.glob(os.path.join(sp, 'nvidia', '*', 'lib'))}
os.environ['LD_LIBRARY_PATH'] = ':'.join(sorted(libdirs)) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print(f'{len(libdirs)} nvidia wheel lib dirs on the path')

smoke = ("from faster_whisper import WhisperModel\n"
         "import numpy as np, soundfile as sf\n"
         "sf.write('/tmp/probe.wav', np.zeros(16000, dtype='float32'), 16000)\n"
         "m = WhisperModel('tiny', device='cuda', compute_type='float16')\n"
         "list(m.transcribe('/tmp/probe.wav', language='hi')[0])\n"
         "print('GPU decode OK')\n")
p = subprocess.run([sys.executable, '-c', smoke], capture_output=True, text=True,
                   env=os.environ)
print(p.stdout or p.stderr[-2000:])
# If this failed: !pip install -q nvidia-cudnn-cu12 nvidia-cublas-cu12   then re-run.

In [ ]:
# 6. Point the frozen config at the GPU ----------------------------------------
# size, compute_type and device are part of the ASR cache key, so this cannot silently
# mix GPU results with CPU decodes. large-v3 is the §4.2 pilot's choice and is
# affordable on a GPU, which retires the 'knowingly weakened baseline' threat (§10).
import re, pathlib
p = pathlib.Path('configs/config.yaml'); t = p.read_text()
t = re.sub(r'^(\s*size:).*$', r'\1 large-v3              # §4.2 pilot decision', t, count=1, flags=re.M)
t = re.sub(r'^(\s*compute_type:).*$', r'\1 float16       # GPU precision', t, count=1, flags=re.M)
t = re.sub(r'^(\s*device:).*$', r'\1 cuda               # Kaggle T4', t, count=1, flags=re.M)
p.write_text(t)
!sed -n '/^model:/,/^decode:/p' configs/config.yaml

In [ ]:
# 7. Corpus + self-tests  (~15 min; re-run every session, /kaggle/temp is wiped) -
# Re-cutting is deterministic and the cache is keyed by utterance id, not by file path,
# so restored decodes still hit after the audio is rebuilt.
import os
if not os.path.exists('data/raw/slr104/test/transcripts/text'):
    !mkdir -p data/raw/slr104
    !cd data/raw/slr104 && curl -sL -C - -o Hindi-English_test.tar.gz \
        https://openslr.trmal.net/resources/104/Hindi-English_test.tar.gz \
        && tar -xzf Hindi-English_test.tar.gz
run('src/prepare_slr104.py', tail=6)
# The distributed segments are whole-second windows and are not usable as shipped;
# see report/01_dataset_and_harness.md §2.
run('src/refine_segments.py', '--radius', '2.5', '--lam', '2.0', tail=6)
run('src/make_tiers.py', '--force', tail=5)
run('src/build_syllabus.py', tail=2)
run('src/selftest.py', tail=2)
run('src/selftest_pipeline.py', tail=2)

In [ ]:
# 8. Pilots: §4.3 language, §4.2 model  (~15 min) ------------------------------
run('src/pilots.py', 'language', '--tier', 'tier1', tail=8)
run('src/pilots.py', 'model', '--tier', 'tier1', tail=8)
run('src/apply_pilot_decisions.py', tail=8)
checkpoint('pilots')

In [ ]:
# 9. Baseline gate + headroom (§8.1, §8.4) -------------------------------------
# Read the printed gate. A baseline WER above ~0.60 means inspect
# report/validation_pairs_tier1_B0.md before trusting anything downstream.
run('src/run_matrix.py', 'baseline', '--tier', 'tier1', tail=40)
checkpoint('baseline')

In [ ]:
# 10. Tier-1 tuning: every hyperparameter is chosen here and only here (~1 h) --
run('src/run_matrix.py', 'tune', '--tier', 'tier1', tail=50)
checkpoint('tune')

In [ ]:
# 11. Tier-2 matrix, the reported results (~2-3 h) ----------------------------
# Six decodes plus two ablation decodes; every M3 row and combination is free, and G
# re-uses decodes it has already paid for (§9.2). Checkpointed per stage above.
run('src/run_matrix.py', 'matrix', '--tier', 'tier2', tail=60)
checkpoint('matrix_tier2')

In [ ]:
# 12. Results ------------------------------------------------------------------
run('src/status.py', tail=60)
from IPython.display import Markdown, Image, display
import pathlib
for f in ('report/results_tier2.md', 'report/results_tier3.md'):
    if pathlib.Path(f).exists():
        display(Markdown(pathlib.Path(f).read_text()))
for f in sorted(pathlib.Path('report/figures').glob('*.png')):
    print(f); display(Image(str(f)))

## Tier 3 — run this in a *separate* session

The final confirmation decodes the complete 5.18-hour test set twice (baseline and the one
best system), roughly 2–3 hours on a T4. Attach this notebook's output first so the Tier-1
and Tier-2 decodes are restored rather than repeated, set `best` from the Tier-2 table, and
run the cell below instead of cells 9–11.

In [ ]:
# 13. Tier 3 (separate session) ------------------------------------------------
best = 'G'   # set from report/results_tier2.md: G, M2, M1, M2+M3a or M3a
run('src/run_matrix.py', 'final', '--tier', 'tier3', '--best', best, tail=40)
checkpoint('final_tier3')
run('src/make_setup_section.py', tail=3)
run('src/repro.py', tail=20)
checkpoint('done')